In [ ]:
from visualization.vis_eeg import ch_psd_subplots, ch_psd_overlap, plot_ti_sensors, plot_preprocessing_result
from preprocessing.preprocess_eeg  import load_raw, annot_raw_manual, get_ica, get_ica_sources, plot_ics, apply_ica, find_bad_ics, basic_preproc_raw, muscle_annot_raw_auto
from processing.process_eeg import compute_psd

%load_ext autoreload
%autoreload 2

In [ ]:
test = False
show = True
save = False
load = False
auto_mode = False

# 1. Inspect raw data

## Load data

In [ ]:
cids_by_pid = {
    'test_01': ['RS_pre', 'RS_cTBS', 'SpaNav_task2'],
    '02': ['RS_EO', 'RS_EC', 'task_cTBS', 'task_HF', 'task_iTBS'],
    '03': ['RS_pre_EO', 'block1', 'block2'],
    '04': ['RS_pre_EO', 'RS_pre_EC', 'block1', 'block2', 'block3', 'block4', 'block5', 'block6', 'RS_post_EO', 'RS_post_EC', 'impedances'],
    '05': ['RS_pre_EO', 'RS_pre_EC', 'block1', 'block2', 'block3', 'block4', 'block5', 'block6', 'RS_post_EO', 'RS_post_EC', 'impedances'],
}

In [ ]:
pid = '04'
cid = cids_by_pid[pid][3]
cid

## Annotate noisy channels and segments

### option a) Load raw containing all annotations previously set

In [ ]:
if load:
    raw_start = load_raw(pid, cid, annotated_bads=True, test=test)

### If not load: first annotate automatically, then manually

In [ ]:
if not load:
    raw_no_annot = load_raw(pid, cid, annotated_bads=False, test=test)
    raw_start = muscle_annot_raw_auto(raw_no_annot, show=show) 

In [ ]:
scroll_rec_kwargs = {
    # 'duration': 40, 
    'duration': raw_start.times[-1] if raw_start.times[-1] < 230 else raw_start.times[-1]//4,
    'n_channels': len(raw_start.ch_names),
    'clipping': None,
    'scalings': {'eeg': 10e-5},
}

In [ ]:
if not auto_mode:
    # %matplotlib qt6
    %matplotlib inline
    raw_start.plot(**scroll_rec_kwargs, picks='eeg')

#### Annotate manually (and save object with all annotations if save=True!)

In [ ]:
%matplotlib inline
bad_chs = ['O1']
bad_seg = {'bad_seg_starts': [], 'bad_seg_lens': []}
raw_start = annot_raw_manual(raw_start, bad_chs=bad_chs, pid=pid, cid=cid, save=save, **bad_seg)

## Compute and plot raw PSD

In [ ]:
raw_psds = compute_psd(raw_start, fmax=200, test=test)

In [ ]:
# Plot each channel PSD as subplot
show_chs = True
ch_psd_subplots(raw_start, raw_psds, cid=cid, pid=pid, test=test, save=save, show=show_chs, file_name_pref='raw')

In [ ]:
# Plot overlapping channels' PSDs 
if show:
    %matplotlib qt6
else:
    %matplotlib inline
ch_psd_overlap(raw_start, raw_psds, cid=cid, pid=pid, save=save, show=show, file_name_pref='raw')

In [ ]:
if not auto_mode:
    # Interactive plot PSD by channels on topography-location
    %matplotlib qt6
    raw_psds.plot_topo(color='k', axis_facecolor='w', fig_facecolor='w', dB=True)  # does not show 'bads' !

# 2. Perform ICA

### Run ICA

In [ ]:
n_components_based_on_var = False
n_components = 60 if load else None

In [ ]:
ica, raw_used_for_ica = get_ica(pid, cid, raw_start, n_components_based_on_var, save=save, load=load, n_components=n_components)

In [ ]:
ica_sources = get_ica_sources(ica, raw_used_for_ica, pid, cid, load=load, save=save)

## Inspect ICs

In [ ]:
%matplotlib inline
plot_ics(ica, raw_used_for_ica, pid, cid, test=test, save=save, show=False)

In [ ]:
plot_ti_sensors(raw_used_for_ica.info, pid, save=save, show=show)

In [ ]:
if not auto_mode:
    %matplotlib qt6
    ica.plot_sources(raw_used_for_ica)

In [ ]:
eog_ics = []  # []  # set to see whether specific ICs are EOG components
muscle_ics = []  # []  # set to see whether specific ICs are muscular components
bad_ics = find_bad_ics(ica, raw_used_for_ica, eog_ics, muscle_ics, pid, cid, save=save, show=False)

## Exclude artifact-ICs and reconstruct signal 

In [ ]:
ics_to_exclude = ica.exclude if load else bad_ics
iclean_raw = apply_ica(ica, raw_start, ics_to_exclude=ics_to_exclude, pid=pid, cid=cid, save=save, load=load)

# 3. Apply other basic cleaning/preprocessing steps

In [ ]:
prepro_raw = basic_preproc_raw(pid, cid, raw_rec_start=iclean_raw, load=load, save=save)

## Re-compute and -plot raw PSD
Verify whether topographies change across frequencies (so whether average referencing managed to remove the 1/f noise)

In [ ]:
clean_psds = compute_psd(prepro_raw, test=test, fmin=prepro_raw.info['highpass'], fmax=prepro_raw.info['lowpass'])

In [ ]:
# Plot overlapping channels' PSDs 
if show:
    %matplotlib qt6
else:
    %matplotlib inline
ch_psd_overlap(prepro_raw, clean_psds, cid=cid, pid=pid, save=save, show=True, file_name_pref='prepro')

In [ ]:
# Plot PSD before vs. after preprocessing pipeline
plot_preprocessing_result(raw_start, prepro_raw, pid, cid, save=save, show=show)

# 4. Visualize final clean data

In [ ]:
scroll_final_kwargs = {
    'duration': 30, 
    'n_channels': len(prepro_raw.ch_names),
    'clipping': None,
    'scalings': {'eeg': 6e-5},
}

In [ ]:
%matplotlib qt6
prepro_raw.plot(**scroll_final_kwargs, picks='eeg')